# T0: Environment Check & Timing
Purpose: Verify Colab GPU, install dependencies, time one forward pass for SD VAE and CompressAI.
Runtime: GPU (T4). Estimated time: ~5 minutes.
Inputs: None (downloads public models automatically).
Outputs: `timing.json` with GPU name, model load times, inference times per batch size.
Return: `results_T0_v1_<timestamp>.zip` → place in `incoming/T0/`


In [ ]:
import subprocess, sys, os, time, json, hashlib
from pathlib import Path

# Mount Drive and set working folder
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/watermark_project')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Print GPU info
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
gpu_info = result.stdout.strip()
print('GPU:', gpu_info)

# Install pinned deps
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'diffusers', 'transformers', 'accelerate',
    'compressai', 'lpips', 'scipy', 'scikit-image', 'pandas', 'tqdm'
], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
import compressai.zoo as zoo
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
timing = {'gpu': gpu_info}

# Load CompressAI q=1
t0 = time.time()
ca_model = zoo.bmshj2018_factorized(quality=1, pretrained=True).eval().to(device)
timing['compressai_load_s'] = time.time() - t0

# Time inference
dummy = torch.rand(4, 3, 256, 256).to(device)
with torch.no_grad():
    t0 = time.time()
    for _ in range(3):
        out = ca_model(dummy)
    timing['compressai_bs4_ms'] = (time.time() - t0) / 3 * 1000
print(f'CompressAI loaded in {timing["compressai_load_s"]:.1f}s, bs=4 inference: {timing["compressai_bs4_ms"]:.0f}ms')
del ca_model


In [ ]:
from diffusers import AutoencoderKL

t0 = time.time()
vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-mse').eval().to(device)
timing['sd_vae_load_s'] = time.time() - t0

dummy = torch.rand(2, 3, 256, 256).to(device) * 2 - 1  # [-1, 1]
with torch.no_grad():
    t0 = time.time()
    for _ in range(3):
        z = vae.encode(dummy).latent_dist.mean
        rec = vae.decode(z).sample
    timing['sd_vae_bs2_ms'] = (time.time() - t0) / 3 * 1000
print(f'SD VAE loaded in {timing["sd_vae_load_s"]:.1f}s, bs=2 inference: {timing["sd_vae_bs2_ms"]:.0f}ms')
del vae


In [ ]:
import zipfile
from datetime import datetime

timing['timestamp'] = datetime.utcnow().isoformat()

out_path = Path('/tmp/timing.json')
with open(out_path, 'w') as f:
    json.dump(timing, f, indent=2)
print('Timing:', json.dumps(timing, indent=2))

# Create return package
ts = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
zip_path = DRIVE_DIR / f'results_T0_v1_{ts}.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    zf.write(out_path, 'timing.json')
    # Write a simple manifest
    manifest = {'task': 'T0', 'version': 'v1', 'files': ['timing.json'], 'timing': timing}
    zf.writestr('manifest.json', json.dumps(manifest, indent=2))
print(f'Package saved to: {zip_path}')
print('Return this file: place it in incoming/T0/')
